### Code for estimating 'state of hyrdogen' in PEM cell

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Locate the CSV file
possible_paths = [
    Path("data/PEM_test/current_sweep/PEM_polarization_characteristics.csv"),
    Path("../data/PEM_test/current_sweep/PEM_polarization_characteristics.csv"),
    Path("../../data/PEM_test/current_sweep/PEM_polarization_characteristics.csv"),
    Path("PEM_polarization_characteristics.csv"),
    Path("/mnt/data/PEM_polarization_characteristics.csv")
]

csv_path = None
for path in possible_paths:
    if path.exists():
        csv_path = path
        break

if csv_path is None:
    raise FileNotFoundError("Could not find PEM_polarization_characteristics.csv")

df = pd.read_csv(csv_path)

df["timestamp"] = pd.to_datetime(df["timestamp"])
df["time_s"] = (df["timestamp"] - df["timestamp"].iloc[0]).dt.total_seconds()

print(f"Loaded file: {csv_path}")
print(f"Number of samples: {len(df)}")
print("\nAvailable columns:")
print(df.columns.tolist())

print("\nDetected scenarios:")
display(df[["scenario", "mode"]].drop_duplicates())

Loaded file: ..\..\data\PEM_test\current_sweep\PEM_polarization_characteristics.csv
Number of samples: 2892

Available columns:
['timestamp', 'scenario', 'mode', 'K1', 'K2', 'K3', 'K4', 'K5', 'K6', 'K7', 'ina1_ok', 'ina1_bus_V', 'ina1_current_mA', 'ina1_power_mW', 'ina1_shunt_mV', 'ina2_ok', 'ina2_bus_V', 'ina2_current_mA', 'ina2_power_mW', 'ina2_shunt_mV', 'ina3_ok', 'ina3_bus_V', 'ina3_current_mA', 'ina3_power_mW', 'ina3_shunt_mV', 'ina4_ok', 'ina4_bus_V', 'ina4_current_mA', 'ina4_power_mW', 'ina4_shunt_mV', 'time_s']

Detected scenarios:


,scenario,mode
0,1,Scenario 1: Grid -> Load
24,3,Scenario 3: Grid -> Load; PV -> PEM
977,6,Scenario 6: PEM -> Load


In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Locate the CSV file
possible_paths = [
    Path("data/PEM_test/current_sweep/PEM_polarization_characteristics.csv"),
    Path("../data/PEM_test/current_sweep/PEM_polarization_characteristics.csv"),
    Path("../../data/PEM_test/current_sweep/PEM_polarization_characteristics.csv"),
    Path("PEM_polarization_characteristics.csv"),
    Path("/mnt/data/PEM_polarization_characteristics.csv")
]

csv_path = None
for path in possible_paths:
    if path.exists():
        csv_path = path
        break

if csv_path is None:
    raise FileNotFoundError("Could not find PEM_polarization_characteristics.csv")

df = pd.read_csv(csv_path)

df["timestamp"] = pd.to_datetime(df["timestamp"])
df["time_s"] = (df["timestamp"] - df["timestamp"].iloc[0]).dt.total_seconds()

print(f"Loaded file: {csv_path}")
print(f"Number of samples: {len(df)}")
print("\nAvailable columns:")
print(df.columns.tolist())

print("\nDetected scenarios:")
display(df[["scenario", "mode"]].drop_duplicates())

Loaded file: ..\..\data\PEM_test\current_sweep\PEM_polarization_characteristics.csv
Number of samples: 2892

Available columns:
['timestamp', 'scenario', 'mode', 'K1', 'K2', 'K3', 'K4', 'K5', 'K6', 'K7', 'ina1_ok', 'ina1_bus_V', 'ina1_current_mA', 'ina1_power_mW', 'ina1_shunt_mV', 'ina2_ok', 'ina2_bus_V', 'ina2_current_mA', 'ina2_power_mW', 'ina2_shunt_mV', 'ina3_ok', 'ina3_bus_V', 'ina3_current_mA', 'ina3_power_mW', 'ina3_shunt_mV', 'ina4_ok', 'ina4_bus_V', 'ina4_current_mA', 'ina4_power_mW', 'ina4_shunt_mV', 'time_s']

Detected scenarios:


,scenario,mode
0,1,Scenario 1: Grid -> Load
24,3,Scenario 3: Grid -> Load; PV -> PEM
977,6,Scenario 6: PEM -> Load


In [2]:
# Hydrogen estimation settings

F = 96485.3329          # Faraday constant [C/mol]
Z_H2 = 2                # Electrons per H2 molecule

H2_FULL_ML = 15.6       # Measured full hydrogen volume [mL]
H2_UNUSED_ML = 7.6      # Hydrogen left when voltage collapsed [mL]
H2_USABLE_ML = H2_FULL_ML - H2_UNUSED_ML

CHARGING_CURRENT_A = 0.200
DISCHARGE_CURRENT_A = 0.050

# Molar volume at approx. room temperature
MOLAR_VOLUME_ML_PER_MOL = 24000  

# Faraday based hydrogen rates
h2_production_rate_ml_s = (
    CHARGING_CURRENT_A / (Z_H2 * F)
) * MOLAR_VOLUME_ML_PER_MOL

h2_consumption_rate_ml_s = (
    DISCHARGE_CURRENT_A / (Z_H2 * F)
) * MOLAR_VOLUME_ML_PER_MOL

print(f"Estimated H2 production rate at 200 mA: {h2_production_rate_ml_s:.4f} mL/s")
print(f"Estimated H2 consumption rate at 50 mA: {h2_consumption_rate_ml_s:.4f} mL/s")
print(f"Full tank volume: {H2_FULL_ML:.1f} mL")
print(f"Usable hydrogen volume: {H2_USABLE_ML:.1f} mL")

Estimated H2 production rate at 200 mA: 0.0249 mL/s
Estimated H2 consumption rate at 50 mA: 0.0062 mL/s
Full tank volume: 15.6 mL
Usable hydrogen volume: 8.0 mL


In [7]:
# Estimate hydrogen state during electrolysis and fuel cell operation

# Keep only PEM related scenarios
#
# Scenario 3:
# PV -> PEM
# Electrolysis operation
#
# Scenario 6:
# PEM -> Load
# Fuel cell operation
df = df[df["scenario"].isin([3, 6])].copy()

# Reset index after filtering
df.reset_index(drop=True, inplace=True)

# Recalculate time so PEM operation starts at zero seconds
df["time_s"] = (
    df["timestamp"] - df["timestamp"].iloc[0]
).dt.total_seconds()

# Extract time vector from dataframe
time_s = df["time_s"].to_numpy()

# Calculate timestep between measurements
dt = np.diff(time_s, prepend=time_s[0])

# First timestep is set to zero
dt[0] = 0

# Detect electrolysis operation
#
# Scenario 3 corresponds to PEM charging
electrolysis_mask = df["scenario"].eq(3)

# Detect fuel cell operation
#
# Scenario 6 corresponds to PEM discharge
fuel_cell_mask = df["scenario"].eq(6)

# Create array for estimated hydrogen volume
#
# Unit: mL
h2_ml = np.zeros(len(df))

# Initial hydrogen level
h2_ml[0] = 0.0

# Integrate hydrogen state over time
for i in range(1, len(df)):

    # Start from previous hydrogen value
    h2_ml[i] = h2_ml[i - 1]

    # Add hydrogen during electrolysis operation
    if electrolysis_mask.iloc[i]:

        h2_ml[i] += (
            h2_production_rate_ml_s
            * dt[i]
        )

    # Remove hydrogen during fuel cell operation
    elif fuel_cell_mask.iloc[i]:

        h2_ml[i] -= (
            h2_consumption_rate_ml_s
            * dt[i]
        )

    # Limit hydrogen state to physical tank capacity
    h2_ml[i] = np.clip(
        h2_ml[i],
        0,
        H2_FULL_ML
    )

# Store estimated hydrogen volume in dataframe
df["h2_estimated_ml"] = h2_ml

# Hydrogen state relative to full tank capacity
df["h2_state_total_percent"] = (
    100
    * df["h2_estimated_ml"]
    / H2_FULL_ML
)

# Calculate discharge available hydrogen
#
# Hydrogen below 7.6 mL still exists physically in the tank,
# but could not be converted into usable electrical output
# during the constant 50 mA discharge experiment
df["h2_discharge_available_ml"] = np.clip(
    df["h2_estimated_ml"] - H2_UNUSED_ML,
    0,
    H2_USABLE_ML
)

# Relative discharge available hydrogen
df["h2_discharge_available_percent"] = (
    100
    * df["h2_discharge_available_ml"]
    / H2_USABLE_ML
)

# Show first rows of calculated values
display(
    df[
        [
            "time_s",
            "scenario",
            "mode",
            "h2_estimated_ml",
            "h2_state_total_percent",
            "h2_discharge_available_ml",
            "h2_discharge_available_percent"
        ]
    ].head(10)
)

# Show last rows of calculated values
display(
    df[
        [
            "time_s",
            "scenario",
            "mode",
            "h2_estimated_ml",
            "h2_state_total_percent",
            "h2_discharge_available_ml",
            "h2_discharge_available_percent"
        ]
    ].tail(10)
)

,time_s,scenario,mode,h2_estimated_ml,h2_state_total_percent,h2_discharge_available_ml,h2_discharge_available_percent
0,0.0,3,Scenario 3: Grid -> Load; PV -> PEM,0.000000,0.000000,0.0,0.0
1,1.0,3,Scenario 3: Grid -> Load; PV -> PEM,0.024874,0.159450,0.0,0.0
2,2.0,3,Scenario 3: Grid -> Load; PV -> PEM,0.049748,0.318901,0.0,0.0
3,3.0,3,Scenario 3: Grid -> Load; PV -> PEM,0.074623,0.478351,0.0,0.0
4,4.0,3,Scenario 3: Grid -> Load; PV -> PEM,0.099497,0.637801,0.0,0.0
5,5.0,3,Scenario 3: Grid -> Load; PV -> PEM,0.124371,0.797252,0.0,0.0
6,6.0,3,Scenario 3: Grid -> Load; PV -> PEM,0.149245,0.956702,0.0,0.0
7,7.0,3,Scenario 3: Grid -> Load; PV -> PEM,0.174120,1.116152,0.0,0.0
8,8.0,3,Scenario 3: Grid -> Load; PV -> PEM,0.198994,1.275602,0.0,0.0
9,10.0,3,Scenario 3: Grid -> Load; PV -> PEM,0.248742,1.594503,0.0,0.0


,time_s,scenario,mode,h2_estimated_ml,h2_state_total_percent,h2_discharge_available_ml,h2_discharge_available_percent
2730,3014.0,6,Scenario 6: PEM -> Load,2.267404,14.534639,0.0,0.0
2731,3015.0,6,Scenario 6: PEM -> Load,2.261185,14.494776,0.0,0.0
2732,3016.0,6,Scenario 6: PEM -> Load,2.254966,14.454913,0.0,0.0
2733,3017.0,6,Scenario 6: PEM -> Load,2.248748,14.415051,0.0,0.0
2734,3018.0,6,Scenario 6: PEM -> Load,2.242529,14.375188,0.0,0.0
2735,3019.0,6,Scenario 6: PEM -> Load,2.236311,14.335326,0.0,0.0
2736,3020.0,6,Scenario 6: PEM -> Load,2.230092,14.295463,0.0,0.0
2737,3021.0,6,Scenario 6: PEM -> Load,2.223874,14.255601,0.0,0.0
2738,3022.0,6,Scenario 6: PEM -> Load,2.217655,14.215738,0.0,0.0
2739,3023.0,6,Scenario 6: PEM -> Load,2.211437,14.175875,0.0,0.0
